In [2]:
%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject


In [3]:
# =============================================================================
# CELL 1: ALL IMPORTS
# =============================================================================

from __future__ import annotations

import fnmatch
import logging
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

from src.Config import get_settings
from src.Utils import (
    MultiEncodingTextLoader,
    _read_file,
    _to_western_digits,
    norm_regu,
    reg,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

print("✓ All imports complete")

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports complete


In [4]:
# =============================================================================
# CELL 2: MODULE-LEVEL CONSTANTS
# =============================================================================

# Article-boundary regex — mirrors EnhancedBaseLawExtractor.ARTICLE_MARK_RE
_ARTICLE_BOUNDARY_RE = re.compile(
    r"""(?m)^\s*(?:المادة|مادة)\s*(?:\(\s*)?"""
    r"""(?P<num>[0-9٠-٩]+(?:\s*(?:مكرر(?:[اأإآى])?)?)?"""
    r"""(?:\s*\(\s*[اأإآA-Za-z]\s*\))?)(?:\s*\))?"""
    r"""\s*[:：\-–\s]""",
    re.VERBOSE,
)

_ARABIC_INDIC_TRANS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

# Table files are kept as large single chunks; articles get per-article splitting
_table_splitter = CharacterTextSplitter(
    separator="\n\n\n\n", chunk_size=500_000, chunk_overlap=0
)

print("✓ Constants defined")

✓ Constants defined


In [5]:
# =============================================================================
# CELL 3: HELPER FUNCTIONS
# =============================================================================

def _normalize_article_no(token: str) -> str:
    token = token.translate(_ARABIC_INDIC_TRANS).strip()
    token = re.sub(r"\s+", " ", token)
    token = re.sub(r"مكر(?:ر[اأإآىًٍَُِّْ]?)", "مكرر", token)
    return token.strip()


def _split_into_articles(full_text: str) -> List[Dict[str, Optional[str]]]:
    """
    Split raw law text into per-article dicts::

        {"article_number": "5", "text": "..."}

    Text before the first article header gets ``article_number=None``
    and is tagged as a preamble by the caller.
    """
    matches  = list(_ARTICLE_BOUNDARY_RE.finditer(full_text))
    articles: List[Dict[str, Optional[str]]] = []

    if not matches:
        # No article markers found → treat entire text as one chunk
        return [{"article_number": None, "text": full_text.strip()}]

    # Preamble before first article
    if matches[0].start() > 0:
        preamble = full_text[: matches[0].start()].strip()
        if preamble:
            articles.append({"article_number": None, "text": preamble})

    for i, m in enumerate(matches):
        start      = m.end()
        end        = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        body       = full_text[start:end].strip()
        article_no = _normalize_article_no(m.group("num") or "")
        if body:
            articles.append({"article_number": article_no, "text": body})

    return articles


print("✓ Helper functions defined")


✓ Helper functions defined


In [6]:
# =============================================================================
# CELL 4: CORPUS CHUNKER
# =============================================================================

class CorpusChunker:
    def __init__(
        self,
        laws_dir:       str | Path = get_settings().DataPath,
        na2d_dir:       str | Path = get_settings().na2d_data_path,
        max_words_book: int        = 400,
        overlap_book:   int        = 50,
        max_words_rule: int        = 300,
        overlap_rule:   int        = 40,
    ) -> None:
        self.laws_dir         = Path(laws_dir)
        self.na2d_dir         = Path(na2d_dir)
        self._book_splitter   = self._make_splitter(max_words_book, overlap_book)
        self._ruling_splitter = self._make_splitter(max_words_rule, overlap_rule)

    # ── internal utilities ────────────────────────────────────────────────

    @staticmethod
    def _make_splitter(max_words: int, overlap: int) -> RecursiveCharacterTextSplitter:
        return RecursiveCharacterTextSplitter(
            chunk_size      = max_words,
            chunk_overlap   = overlap,
            length_function = lambda t: len(t.split()),
            separators      = ["\n\n", "\n", ".", " ", ""],
        )

    @staticmethod
    def _book_title(stem: str) -> str:
        for key, title in norm_regu.BOOK_TITLE_MAP.value.items():
            if key in stem:
                return title
        return stem

    @staticmethod
    def _ruling_metadata(text: str, folder: str, filename: str) -> Dict[str, Any]:
        header = text[:1000]
        meta: Dict[str, Any] = {
            "doc_type":       "ruling",
            "crime_category": folder,
            "source_file":    filename,
        }
        m = reg._CASE_NUM_RE.value.search(header)
        if m:
            meta["case_number"] = _to_western_digits(m.group(1).replace(" ", ""))
            meta["case_year"]   = _to_western_digits(m.group(2))
        m = reg._RULING_DATE_RE.value.search(header)
        if m:
            meta["ruling_date"] = m.group(0).replace("جلسة", "").strip()
        m = reg._CHAMBER_RE.value.search(header)
        if m:
            meta["chamber"] = m.group(1).strip()
        return meta

    def _get_amendment_metadata(
        self, raw_text: str, filename: str
    ) -> Optional[Dict[str, str]]:
        """
        Extract amendment law number, year, date, and type.

        **Filename is the primary source** for law number + year (same logic as
        :class:`AmendmentLawExtractor`).  The text header is used only as a
        fallback because it refers to the *original* law being amended, not
        the amending law.
        """
        law_num: Optional[str] = None
        year:    Optional[str] = None

        # ── Primary: filename ─────────────────────────────────────────────
        # Pattern 1: ..._num{N}_{YYYY}.txt
        m = re.search(r'_num([0-9]+)_([0-9]{4})\.txt$', filename, re.IGNORECASE)
        if m:
            law_num, year = m.group(1), m.group(2)

        # Pattern 2: ..._{N}_{YYYY}.txt
        if not law_num or not year:
            m = re.search(r'_([0-9]+)_([0-9]{4})\.txt$', filename, re.IGNORECASE)
            if m:
                law_num, year = m.group(1), m.group(2)

        # Pattern 3: any 4-digit year + any other number in filename
        if not law_num or not year:
            years = re.findall(r'((?:19|20)\d{2})', filename)
            nums  = [n for n in re.findall(r'(\d+)', filename) if n not in years]
            law_num = nums[-1] if nums else None
            year    = years[-1] if years else None

        # ── Fallback: scan text header ────────────────────────────────────
        if not law_num or not year:
            for m in reg.ORIGINAL_LAW_RE.value.finditer(raw_text):
                try:
                    a = int(_to_western_digits(m.group(1)))
                    b = int(_to_western_digits(m.group(2)))
                    year, law_num = (
                        (str(a), str(b)) if 1800 <= a <= 2100 else (str(b), str(a))
                    )
                    if 1800 <= int(year) <= 2100 and 0 < int(law_num) < 10000:
                        break
                except Exception:
                    continue

        if not law_num or not year:
            return None

        # ── Date ──────────────────────────────────────────────────────────
        date_m = reg._DATE_RE.value.search(raw_text)
        if date_m:
            day   = _to_western_digits(date_m.group(1) or "01").zfill(2)
            month = reg._MONTH_MAP.value.get(date_m.group(2), "01")
            yr    = _to_western_digits(date_m.group(3))
            adate = f"{yr}-{month}-{day}"
        else:
            adate = f"{year}-01-01"

        # ── Amendment type ─────────────────────────────────────────────────
        sample = raw_text[:800]
        if any(w in sample for w in norm_regu.ADDITION_KEYWORDS.value):
            atype = "addition"
        elif any(w in sample for w in norm_regu.DELETION_KEYWORDS.value):
            atype = "deletion"
        else:
            atype = "modification"

        return {
            "law_num":        law_num,
            "year":           year,
            "amendment_date": adate,
            "amendment_type": atype,
        }

    # ── public API ────────────────────────────────────────────────────────

    def get_chunks(self) -> List[Document]:
        """
        Return one :class:`Document` per article / preamble / amended-article
        / table row for every law folder in ``laws_dir``.
        """
        all_docs: List[Document] = []

        for folder in sorted(p for p in self.laws_dir.iterdir() if p.is_dir()):
            if folder.name not in norm_regu.FOLDER_TO_LAW_KEY.value:
                continue

            law_key   = norm_regu.FOLDER_TO_LAW_KEY.value[folder.name]
            law_title = norm_regu.LAW_KEY_TO_TITLE.value.get(law_key, law_key)

            # ── main law files (non-amendment) ────────────────────────────
            for fp in sorted(f for f in folder.glob("*.txt") if not fnmatch.fnmatch(f.name.lower(), "new*")):
                raw = MultiEncodingTextLoader(fp).load()
                if not raw:
                    continue
                full_text = raw[0].page_content

                if "tables" in fp.name.lower():
                    # Tables: one large chunk per logical table block
                    docs = _table_splitter.create_documents(
                        texts     = [full_text],
                        metadatas = [{
                            "law_id":         law_key,
                            "law_title":      law_title,
                            "chunk_type":     "table",
                            "article_number": None,
                            "source_file":    fp.name,
                        }],
                    )
                    all_docs.extend(docs)
                    logger.info(f"  {fp.name} [table] → {len(docs)} chunks")

                else:
                    # Articles: exactly one Document per article
                    articles = _split_into_articles(full_text)
                    for art in articles:
                        chunk_type = (
                            "preamble" if art["article_number"] is None else "article"
                        )
                        all_docs.append(Document(
                            page_content = art["text"],
                            metadata     = {
                                "law_id":         law_key,
                                "law_title":      law_title,
                                "chunk_type":     chunk_type,
                                "article_number": art["article_number"],
                                "source_file":    fp.name,
                            },
                        ))
                    logger.info(f"  {fp.name} [article] → {len(articles)} chunks")

            # ── amendment files (new*) ─────────────────────────────────────
            for af in sorted(folder.glob("new*.txt")):
                raw = MultiEncodingTextLoader(af).load()
                if not raw:
                    continue
                full_text = raw[0].page_content
                meta      = self._get_amendment_metadata(full_text, af.name)
                if not meta:
                    logger.warning(f"  [SKIP] {af.name} — no amendment metadata")
                    continue

                articles = _split_into_articles(full_text)
                for art in articles:
                    all_docs.append(Document(
                        page_content = art["text"],
                        metadata     = {
                            "law_id":                law_key,
                            "law_title":             law_title,
                            "chunk_type":            "amended_article",
                            "article_number":        art["article_number"],
                            "amendment_law_number":  meta["law_num"],
                            "amendment_date":        meta["amendment_date"],
                            "amendment_type":        meta["amendment_type"],
                            "source_file":           af.name,
                        },
                    ))
                logger.info(f"  {af.name} [amendment] → {len(articles)} chunks")

        logger.info(f"✓ laws total: {len(all_docs)} chunks")
        return all_docs

    def get_na2d_chunks(self) -> Dict[str, List[Document]]:
        """
        Return ruling and legal-principle chunks from the Na2d corpus.

        Returns a dict with two keys:

        * ``"rulings"``    — court rulings, split by ``_ruling_splitter``
        * ``"principles"`` — legal-principle books, split by ``_book_splitter``
        """
        ruling_docs:    List[Document] = []
        principle_docs: List[Document] = []

        for item in sorted(self.na2d_dir.iterdir()):

            if item.is_file() and item.suffix.lower() == ".txt":
                # Legal-principle book
                text = _read_file(item)
                if not text:
                    continue
                chunks = self._book_splitter.create_documents(
                    texts     = [text.strip()],
                    metadatas = [{
                        "doc_type":   "principle",
                        "book_title": self._book_title(item.stem),
                        "source_file": item.name,
                    }],
                )
                principle_docs.extend(chunks)
                logger.info(f"  [BOOK] {item.name} → {len(chunks)} chunks")

            elif item.is_dir():
                # Court rulings folder
                folder_count = 0
                for fp in sorted(item.glob("*.txt")):
                    text = _read_file(fp)
                    if not text:
                        continue
                    chunks = self._ruling_splitter.create_documents(
                        texts     = [text.strip()],
                        metadatas = [self._ruling_metadata(text, item.name, fp.name)],
                    )
                    ruling_docs.extend(chunks)
                    folder_count += len(chunks)
                logger.info(f"  [RULINGS] {item.name} → {folder_count} chunks")

        logger.info(
            f"✓ na2d: rulings={len(ruling_docs)} | principles={len(principle_docs)}"
        )
        return {"rulings": ruling_docs, "principles": principle_docs}

In [7]:
# =============================================================================
# ENTRY POINTS
# =============================================================================

def get_chunks() -> List[Document]:
    """Convenience wrapper — returns law chunks from a default :class:`CorpusChunker`."""
    return CorpusChunker().get_chunks()


def get_na2d_chunks() -> Dict[str, List[Document]]:
    """Convenience wrapper — returns Na2d chunks from a default :class:`CorpusChunker`."""
    return CorpusChunker().get_na2d_chunks()

In [8]:
chunks = get_chunks()

2026-03-21 20:09:39,209 - INFO -   3okobat.txt [article] → 538 chunks
2026-03-21 20:09:39,221 - INFO -   new_3okobat_num36_2020.txt [amendment] → 1 chunks
2026-03-21 20:09:39,223 - INFO -   new_3okobat_num6_2020.txt [amendment] → 2 chunks
2026-03-21 20:09:39,235 - INFO -   قانون-مكافحة-غسل-الامول_ultra_clean.txt [article] → 21 chunks
2026-03-21 20:09:39,236 - INFO -   asle7a.txt [article] → 52 chunks
2026-03-21 20:09:39,237 - INFO -   asle7a_tables.txt [table] → 1 chunks
2026-03-21 20:09:39,242 - INFO -   dostor_masr.txt [article] → 252 chunks
2026-03-21 20:09:39,243 - INFO -   drugs.txt [article] → 69 chunks
2026-03-21 20:09:39,244 - INFO -   drugs_tables.txt [table] → 1 chunks
2026-03-21 20:09:39,249 - INFO -   egra2at_gena2ya.txt [article] → 511 chunks
2026-03-21 20:09:39,251 - INFO -   erhab.txt [article] → 55 chunks
2026-03-21 20:09:39,252 - INFO -   قانون-الطوارئ_ultra_clean.txt [article] → 21 chunks
2026-03-21 20:09:39,253 - INFO -   tech.txt [article] → 46 chunks
2026-03-21 20:

In [9]:
print(len(chunks))

1570


In [10]:
from IPython.display import display , Markdown

def dis(chunk):
    print(chunk.metadata)
    print(display(Markdown(chunk.page_content)))


In [11]:
dis(chunk=chunks[0])

{'law_id': 'penal_code', 'law_title': 'قانون العقوبات', 'chunk_type': 'preamble', 'article_number': None, 'source_file': '3okobat.txt'}


**آخر تعديل: 15 أغسطس 2021 بالقانون 141 لسنة 2021 **
القانون رقم 58 لسنة 1937
___________________________________________

None


In [12]:
tables = [chunk.page_content for chunk in chunks if chunk.metadata['chunk_type']=='table']

In [13]:
len(tables)

2

In [14]:
def table_dis(chunk):
    print(display(Markdown(chunk)))

In [15]:
for i,table_file in enumerate(tables) : 
    table_dis(chunk=table_file)

جدول رقم (1)
بيان الأسلحة البيضاء
1 – السيوف (عدا سيوف المبارزة).
2 – السونكى.
3 – الخنجر.
4 – الأقواس والسهام.
5 – المطاوى قرن الغزال.
6 – السواطير، والسكاكين “عدا ما يستخدم منها فى الأغراض المنزلية أو الفندقية حال التعامل معها بمسوغ قانونى”.
7 – البلط، والجنازير، والسنج، والقواطع (الكترات)، والشفرات، والروادع الشخصية، وعصى الصدمات، والدونكات، وأية أداة أخرى تستخدم فى الاعتداء على الأشخاص دون أن يوجد لحملها أو إحرازها أو حيازتها مسوغ قانونى، أو مبرر من الضرورة المهنية أو الحرفية.
8 – الملكمة الحديد (البونية).
أية أجهزة أو أدوات أو آلات أو منتجات، أيًا ما كان شكلها، تحتوي على أسلحة بيضاء.
القيود الحديدية، والصديرى والخوذة الواقيتان من الرصاص.
___________________________________________
جدول رقم (2)
1 – الأسلحة النارية غير المششخنة
2 –الأسلحة النارية ذات الماسورة المصقولة من الداخل.
___________________________________________
جدول رقم (3)
الأسلحة المششخنة (2)
وينقسم هذا النوع إلى قسمين:
القسم الأول
(أ) المسدسات فردية الإطلاق.
(ب) البنادق المششخنة ذات التعمير اليدوى والتي تطلق طلقة طلقة.
القسم الثاني
(أ) المدافع والمدافع الرشاشة.
(ب) البنادق المششخنة النصف آلية والآلية سريعة الطلقات.
(جـ) المسدسات سريعة الطلقات.
أية أجهزة أو أدوات أو آلات أو منتجات، أيًا ما كان شكلها، تحتوى على أسلحة نارية
___________________________________________
الجدول رقم (4)
الأجزاء الرئيسية للأسلحة النارية
أولاً – بالنسبة للبنادق ذات الماسورة المصقولة من الداخل:
1 – الجسم المعدنى.
2 – الماسورة.
ثانياً – بالنسبة للبنادق المششخنة والنصف آلية:
1 – الجسم المعدنى (الظرف).
2 – الماسورة.
3 – الترباس ومجموعته.
ثالثاً – بالنسبة للمسدسات بكافة أنواعها:
(أ) مسدس بخزنة:
1 – الجسم المعدنى.
2 – المنزلق.
3 – الماسورة.
(ب) مسدس بساقية:
1 – الجسم المعدنى.
2 – الأكرة (الساقية).
رابعاً – بالنسبة للمدافع والرشاشات والبنادق الآلية:
(أ) المدافع والرشاشات:
1 – الجسم المعدنى.
2 – الماسورة.
(ب) البنادق الآلية:
1 – الجسم المعدنى.
2 – الماسورة.
3 – الترباس ومجموعته.
الجدول رقم (5) (4)
مسدسات وبنادق الصوت وضغط الهواء وضغط الغاز وذخائرها.

None


الجدول رقم (1)
المواد المعتبرة مخدرة
القسم الأول
1. كوكايين (Cocaine)
الاسم الكيميائي: استر المثيل لبنزويل إيكجونين 
Chemical Name: Methyl ester of benzyolecgonine
الوصف: 
كافة مستحضرات الكوكايين المدرجة في دساتير الأدوية والتي تحتوي على أكثر من 0.1% من الكوكايين سواء صنعت من أوراق الكوكا (خلاصتها السائلة أو صبغتها) أو من الكوكايين ومخففات الكوكايين في مادة غير فعالة أو صلبة أيًا كانت درجة تركيزها.
---
2. هيروين (Heroin)
الاسم الكيميائي: ثنائي أسيتيل مورفين 
Chemical Name: Diacetylmorphine (Acetomorphine – Diamorphine)
الوصف: 
ثنائي أسيتيل مورفين بذاته أو مخلوطًا أو مخففًا في أي مادة كانت درجة تركيزها وبأي نسبة.
---
3. 6-أحادي أسيتيل المورفين
Chemical Name: 6-MAM (6-Mono Acetyl Morphine)
---
القسم الثاني
1. إيتورفين (Etorphine)
الاسم الكيميائي: 7،8-ثنائي إيدرو-7 ألفا-[1-(آر)-هيدروكسي-1-مثيل بيوتيل] أوكسي-مثيل 6،14 إندواثينو مورفين 
Chemical Name: 7,8-dihydro-7α-[1(R)-hydroxy-1-methylbutyl]-O-methyl-6,14-endoethenomorphine
الأسماء التجارية: Immobilon-M 99
---
2. إثيل مثيل الثيامبيوتين (Ethylmethylthiambutene)
الاسم الكيميائي: 3-إثيل مثيل أمينو-1،1-ثنائي (2-ثينيل)-1-بيوتين 
Chemical Name: 3-dimethylamino-1,1-di-(2-thienyl)-1-butene
الأسماء التجارية: Emethibutin – Ethylmethiambutene
---
3. أسيتيل ميثادول (Acetylmethadol)
الاسم الكيميائي: 3-أسيتوكسي-6 ثنائي مثيل أمينو 4،4 ثنائي فنيل هيبتان 
Chemical Name: 3-acetoxy-6-dimethylamino-4,4-diphenylheptane
الأسماء التجارية: Amidol acetate – Methadyl acetate
---
 4. أسيتورفين (Acetorphine)
الاسم الكيميائي: 3 أوكسي-أسيتيل-8،7 ثنائي هيدرو-7 ألفا-[1 (ر)-هيدروكسي-1-مثيل بيوتيل] 6 أوكسي-مثيل-14،6 إندواثينو مورفين 
Chemical Name: O-acetyl-7,8-dihydro-7α-[1(R)-hydroxy-1-methylbutyl] O-methyl-6,14-endoethenomorphine
الأسماء التجارية: M183
---
 5. إيكجونين (Ecgonine)
الاسم الكيميائي: (−)-3-هيدروكسي تروبان-2-كاربوكسيلات 
Chemical Name: (−)-3-Hydroxytropane-2-Carboxylate
الأسماء التجارية: Laevo-ecgonine
---
 6. أوكسيكودون (Oxycodone)
الاسم الكيميائي: 14-هيدروكسي ثنائي هيدروكودينون 
Chemical Name: 14-hydroxydihydrocodeinone (Dihydrohydroxycodeinone)
الأسماء التجارية: Codenon – Dihydrone – Eucodal
---
 7. أوكسيمورفون (Oxymorphone)
الاسم الكيميائي: 14-هيدروكسي ثنائي هيدرومورفينون 
Chemical Name: 14-hydroxydihydromorphinone (Dihydrohydroxymorphinone)
الأسماء التجارية: Numorphan-5501
---
 8. أوكسيد-ن-المورفين (Morphine-N-Oxide)
الوصف: 
وكذا المركبات المورفينية الأخرى ذات الأزوت الخماسي التكافؤ
الأسماء التجارية: Genomorphine, Codeine-N-Oxide
---
 9. الأفيون (Opium)
الوصف: 
ويشمل الأفيون الخام والأفيون الطبي والأفيون المحضر بجميع مسمياتهم، وكافة مستحضرات الأفيون المدرجة أو غير المدرجة في دساتير الأدوية والتي تحتوي على أكثر من 0.2% من المورفين ومخففات الأفيون في مادة غير فعالة سائلة أو صلبة أيًا كانت درجة تركيزها.
---
 10. ألفا برودين (Alphaprodine)
الاسم الكيميائي: ألفا-1،3 ثنائي مثيل-4-فنيل-4-بروبيونوكسي بيبريدين 
Chemical Name: α-1,3-dimethyl-4-phenyl-4-propionoxypiperidine
الأسماء التجارية: GF-21, Nisentil, Prisilidene
---
 11. ألفا أسيتيل ميثادول (Alphacetylmethadol)
الاسم الكيميائي: ألفا-3-أسيتوكسي-6-ثنائي مثيل أمينو-4،4-ثنائي فنيل هيبتان 
Chemical Name: α-3-acetoxy-6-dimethylamino-4,4-diphenylheptane
الأسماء التجارية: N.I.H 2953
---
 12. ألفا ميبرودين (Alphameprodine)
الاسم الكيميائي: ألفا-3-إثيل-1-مثيل-4-فنيل-4-بروبيونوكسي بيبريدين 
Chemical Name: α-3-ethyl-1-methyl-4-phenyl-4-propionoxypiperidine
الأسماء التجارية: Nu2-1932
---
 13. ألفا ميثادول (Alphamethadol)
الاسم الكيميائي: ألفا-6-ثنائي مثيل أمينو-4،4-ثنائي فنيل-3-هيبتانول 
Chemical Name: α-6-dimethylamino-4,4-diphenyl-3-heptanol
---
 14. أليل برودين (Allylprodine)
الاسم الكيميائي: 3-أليل-1-مثيل-4-فنيل-4-بروبيونوكسي بيبريدين 
Chemical Name: 3-allyl-1-methyl-4-phenyl-4-propionoxypiperidine
الأسماء التجارية: Alporidine-N.I.H-7440
---
 15. أمفيتامين (Amfetamine)
الاسم الكيميائي: (±)-2-أمينو-1 فنيل بروبان 
Chemical Name: (±)-2-amino-1-phenylpropane
الوصف: 
بذاته وأملاحه بذاتها في جميع أشكالها الصيدلية المختلفة.
الأسماء التجارية: Anorexine, Actedron, Benzedrine, Aktedron
ملاحظة: ليفو أمفيتامين لا يعتبر مادة مخدرة.
---
 16. أموباربيتال (Amobarbital)
الاسم الكيميائي: 5-إثيل-5-(3-مثيل بيوتيل) حمض باربيتوريك 
Chemical Name: 5-ethyl-5-(3-methylbutyl) barbituric acid
الوصف: 
بذاته وأملاحه بذاتها في جميع أشكالها الصيدلية المختلفة.
الأسماء التجارية: Amytal
---
 17. أنيليريدين (Anileridine)
الاسم الكيميائي: 1-بارا-أمينوفين إثيل-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 1-para-aminophenethyl-4-phenylpiperidine-4-carboxylic acid ethyl ester
الأسماء التجارية: Leritine-MK-89-WIN13707
---
 18. إيثوكسيريدين (Etoxeridine)
الاسم الكيميائي: 1-[2-(2-هيدروكسي إثوكسي) إثيل]-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 1-[2-(2-hydroxyethoxy)ethyl]-4-phenylpiperidine-4-carboxylic acid ethyl ester
الأسماء التجارية: Atenотax, Atenos, Carbetidine, U.C2072
---
 19. إيتونيتازين (Etonitazene)
الاسم الكيميائي: 1-ثنائي إثيل أمينو إثيل-2-بارا-إثوكسي بنزيل-5-نيتروبنزيميدازول 
Chemical Name: 1-diethylaminoethyl-2-para-ethoxybenzyl-5-nitrobenzimidazole
الأسماء التجارية: N.I.H-7607
---
 20. هيدروكودون (Hydrocodone)
الاسم الكيميائي: ثنائي هيدروكودينون 
Chemical Name: Dihydrocodeinone
الأسماء التجارية: Ambenyl, Calmodid, Dicodide, Diconone, Biocodone
---
 21. هيدروكسي بيتيدين (Hydroxypethidine)
الاسم الكيميائي: 4-ميتا-هيدروكسي فنيل-1-مثيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 4-meta-hydroxyphenyl-1-methyl piperidine-4-carboxylic acid ethyl ester
---
 22. هيدرومورفون (Hydromorphone)
الاسم الكيميائي: ثنائي هيدرومورفينون 
Chemical Name: Dihydromorphinone
الأسماء التجارية: Laudadin, Dilaudid, Dimorphone
---
 23. هيدرومورفينول (Hydromorphinol)
الاسم الكيميائي: 14-هيدروكسي ثنائي هيدرومورفين 
Chemical Name: 14-hydroxydihydromorphine
الأسماء التجارية: N.I.H-7472
---
 24. أيزوميثادون (Isomethadone)
الاسم الكيميائي: 6-ثنائي مثيل أمينو-5-مثيل-4،4-ثنائي فنيل-3-هيكسانون 
Chemical Name: 6-dimethylamino-5-methyl-4,4-diphenyl-3-hexanone
الأسماء التجارية: Isoadanon, Isoamidone-N.I.H-2880
---
 25. بيثيدين (Pethidine)
الاسم الكيميائي: 1-مثيل-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 1-methyl-4-phenylpiperidine-4-carboxylic acid ethyl ester
الأسماء التجارية: Dolantin, Demerol, Dolosile
---
 26. وسيط البيثيدين ألف (Pethidine Intermediate-A)
الاسم الكيميائي: 4-سيانو-1-مثيل-4-فنيل بيبريدين 
Chemical Name: 4-cyano-1-methyl-4-phenylpiperidine
الأسماء التجارية: Pre-Pethidine
---
 27. وسيط البيثيدين ب (Pethidine Intermediate-B)
الاسم الكيميائي: 4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 4-phenylpiperidine-4-carboxylic acid ethyl ester
الأسماء التجارية: Norpethidine
---
 28. وسيط البيثيدين ج (Pethidine Intermediate-C)
الاسم الكيميائي: 1-مثيل-4-فنيل بيبريدين-4-حمض كاربوكسيليك 
Chemical Name: 1-methyl-4-phenylpiperidine-4-carboxylic acid
الأسماء التجارية: Meperidinic acid
---
 29. بسيلوسيبين (Psilocybine)
الاسم الكيميائي: 3-(2-ثنائي مثيل أمينو إثيل) إندول-4-يل-ثنائي هيدروجين فوسفات 
Chemical Name: 3-(2-dimethylaminoethyl)indol-4-yl dihydrogen phosphate
---
 30. بروبيريدين (Properidine)
الاسم الكيميائي: 1-مثيل-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر أيزوبروبيل 
Chemical Name: 1-methyl-4-phenylpiperidine-4-carboxylic acid isopropyl ester
الأسماء التجارية: Gevelina, Ipropethidine, Isopedine
---
 56. حشيش (Cannabis)
الوصف: 
بجميع أنواعه ومسمياته مثل الكمنجة أو البانجو أو الماريجوانا أو غير ذلك من الأسماء التي قد تطلق عليه، الناتج أو المحضر أو المستخرج من ثمار أو أوراق أو سيقان أو جذور أو راتنج نبات القنب (كنابيس ساتيفا) ذكرًا كان أو أنثى أو الناتج عن تجفيف ثماره أو أزهاره أو أوراقه.
يشمل:
- المستحضرات الجالينوسية للقنب (الخلاصة أو الصبغة)
- المستحضرات التي قاعدتها خلاصة أو صبغة القنب
- مستحضرات راتنج القنب (أي كافة المستحضرات المحتوية على عنصر القنب الفعال أي الراتنج بأي نسبة كانت)
- خلاصة النبات أو أي جزء منه مثل زيت الحشيش
- المساحيق المكونة من كل أو بعض أجزاء نبات الحشيش
- التنجات الناتجة من النبات سواء كانت في صورة نقية أو على شكل خليط أيًا كان نوعه
---
 57. ديكسا مفيتامين (Dexamfetamine)
الاسم الكيميائي: (+)-2-أمينو-1-فنيل بروبان 
Chemical Name: (+)-2-amino-1-phenylpropane / (+)-α-methyl phenethylamine
الأسماء التجارية: Maxiton, Dexedrine
---
 58. دكسترومواميد (Dextromoramide)
الاسم الكيميائي: (+)-4-[2-مثيل-4-أوكسو-3،3-ثنائي فنيل-4-(1-بيروليدينيل) بيوتيل] مورفولين 
Chemical Name: (+)-4-[2-methyl-4-oxo-3,3-diphenyl-4-(1-pyrolidinyl)butyl]morpholine / (+)-3-methyl-2,2-diphenyl-4-morpholino butyryl-pyrrolidine
الأسماء التجارية: Pyrrolamido-N.I.H-7422-SKFD-5137
---
 59. دروتيبانول (Drotebanol)
الاسم الكيميائي: 3،4-ثنائي ميثوكسي-17-مثيل مورفينان-6 بيتا،14-ديول 
Chemical Name: 3,4-dimethoxy-17-methylmorphinan-6β,14-diol
---
 60. ثنائي أمبروميد (Diampromide)
الاسم الكيميائي: ن-[2-(مثيل فين إثيل أمينو) بروبيل] بروبيونانيليد 
Chemical Name: N-[(2-methylphenethylamino)propyl]propionanilide
---
 61. ديزومورفين (Desomorphine)
الاسم الكيميائي: ثنائي هيدرودي أوكسي مورفين 
Chemical Name: Dihydrodeoxymorphine / 4,5-epoxy-3-hydroxy-N-methylmorphinan
الأسماء التجارية: Permonid
---
 62. راسيموراميد (Racemoramide)
الاسم الكيميائي: (±)-4-[2-مثيل-4-أوكسو-3،3-ثنائي فنيل-4-(1-بيروليدينيل) بيوتيل] مورفولين 
Chemical Name: (±)-4-[2-methyl-4-oxo-3,3-diphenyl-4-(1-pyrrolidinyl)butyl]morpholine / (±)-3-methyl-2,2-diphenyl-4-morpholino-butyryl-pyrrolidine
الأسماء التجارية: N.I.H-7421-SKF5137
---
 63. راسيمورفان (Racemorphan)
الاسم الكيميائي: (±)-3-هيدروكسي-ن-مثيل مورفينان 
Chemical Name: (±)-3-hydroxy-N-methylmorphinan
الأسماء التجارية: Citarin, Methorphinan
ملاحظة: ديكستروفان (Dextrophan) لا تعتبر مادة مخدرة.
---
 64. راسيميثورفان (Racemethorphan)
الاسم الكيميائي: (±)-3-ميثوكسي-ن-مثيل مورفينان 
Chemical Name: (±)-3-methoxy-N-methylmorphinan
الأسماء التجارية: Methorphinan-Ro.1-5470
ملاحظة: ديكستروميثورفان (Dextromethorphan) لا تعتبر مادة مخدرة.
---
 65. سيكوباربيتال (Secobarbital)
الاسم الكيميائي: 5-أليل-5-(1-مثيل بيوتيل) حمض باربيتوريك 
Chemical Name: 5-allyl-5-(1-methylbutyl)barbituric acid
الوصف: بذاته وأملاحه بذاتها في جميع أشكالها الصيدلية المختلفة.
الأسماء التجارية: Seconal, Quinalbarbital
---
 66. فينادوكسون (Phenadoxone)
الاسم الكيميائي: 6-مورفولينو-4،4-ثنائي فنيل-3-هيبتانون 
Chemical Name: 6-morpholino-4,4-diphenyl-3-heptanone
الأسماء التجارية: C.B.11, Heptalgin
---
 67. فينازوسين (Phenazocine)
الاسم الكيميائي: 2-هيدروكسي-5،9-ثنائي مثيل-2 فين إثيل-6،7-بنزومورفان 
Chemical Name: 2-hydroxy-5,9-dimethyl-2-phenethyl-6,7-benzomorphan / 1,2,3,4,5,6-hexahydro-8-hydroxy-6,11-dimethyl-3-phenethyl-2,6-methano-3-benzazocine
الأسماء التجارية: Narcidine, Prinadol-N.I.H-7519
---
 68. فينامبروميد (Phenampromide)
الاسم الكيميائي: ن-(1-مثيل-2-بيبيريدينو إثيل) بروبيونانيليد 
Chemical Name: N-(1-methyl-2-piperidinoethyl)propionanilide / N-[2-(1-methylpiperid-2-yl)ethyl]propionanilide
---
 69. فنتانيل (Fentanyl)
الاسم الكيميائي: 1-فين إثيل-4-ن-بروبيونيل أنيلينوبيبريدين 
Chemical Name: 1-phenethyl-4-N-propionylanilinopiperidine
الأسماء التجارية: R.4263, Thalamonial
---
 70. فينوبيريدين (Phenoperidine)
الاسم الكيميائي: 1-(3 هيدروكسي-3-فنيل بروبيل)-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 1-(3-hydroxy-3-phenylpropyl)-4-phenylpiperidine-4-carboxylic acid ethyl ester / 1-phenyl-3-(4-carbethoxy-4-phenyl piperidine)-propanol
الأسماء التجارية: Phenopropidine-R.1406
---
 71. فينومورفان (Phenomorphan)
الاسم الكيميائي: 3-هيدروكسي-ن-فين إثيل مورفينان 
Chemical Name: 3-hydroxy-N-phenethylmorphinan
---
 72. فيوريثيدين (Furethidine)
الاسم الكيميائي: 1-(2-رباعي هيدرو فورفوريل أوكسي إثيل)-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 1-(2-tetrahydro furfuryloxyethyl)-4-phenylpiperidine-4-carboxylic acid ethyl ester
الأسماء التجارية: TA48
---
 73. كلونيتازين (Clonitazene)
الاسم الكيميائي: 2-بارا-كلوربنزيل-1-ثنائي إثيل أمينو إثيل-5-نيتروبنزيميدازول 
Chemical Name: 2-para-chlorobenzyl-1-diethylaminoethyl-5-nitrobenzimidazole
---
 74. كودوكسيم (Codoxime)
الاسم الكيميائي: ثنائي هيدروكودينون-6-كاربوكسي مثيل أوكسيم 
Chemical Name: Dihydrocodeinone-6-carboxymethyloxime
---
 75. كيتوبيميدون (Ketobemidone)
الاسم الكيميائي: 4-ميتا هيدروكسي فنيل-1-مثيل-4-بروبيونيل بيبريدين 
Chemical Name: 4-meta-hydroxyphenyl-1-methyl-4-propionylpiperidine / 4-(3-hydroxyphenyl)-1-methyl-4-propionylpiperidine / 1-methyl-4-metahydroxyphenyl-4-propionylpiperidine
الأسماء التجارية: Cliradon, Ketogan
---
 76. (+)-ليسرجيد ((+)-Lysergide)
الاسم الكيميائي: (+)-ن،ن-ثنائي إثيل ليسرجاميد (د-حمض ليسرجيك ثنائي إثيل أميد) 
Chemical Name: (+)-N,N-diethyllysergamide (d-lysergic acid diethylamide)
الأسماء التجارية: LSD, LSD-25, Delysid
---
 77. ليفورفانول (Levorphanol)
الاسم الكيميائي: (-)-3-هيدروكسي-ن-مثيل مورفينان 
Chemical Name: (-)-3-hydroxy-N-methylmorphinan
الأسماء التجارية: Levorphan, Dromoran-(N.I.H-45900)
ملاحظة: ديكستروفان (Dextrophan) لا تعتبر مادة مخدرة.
---
 78. ليفوفيناسيل مورفان (Levophenacylmorphan)
الاسم الكيميائي: (-)-3-هيدروكسي-ن-فيناسيل مورفينان 
Chemical Name: (-)-3-hydroxy-N-phenacylmorphinan
الأسماء التجارية: (Ro.4-0288)(N.I.H.7525)
---
 79. ليفوموراميد (Levomoramide)
الاسم الكيميائي: (-)-4-[2-مثيل-4-أوكسو-3،3-ثنائي فنيل-4-(1-بيروليدينيل) بيوتيل] مورفولين 
Chemical Name: (-)-4-[2-methyl-4-oxo-3,3-diphenyl-4-(1-pyrrolidinyl)butyl]morpholine / L-3-methyl-2,2-diphenyl-4-morpholino-butyryl-pyrrolidine
---
 80. ليفوميثورفان (Levomethorphan)
الاسم الكيميائي: (-)-3-ميثوكسي-ن-مثيل مورفينان 
Chemical Name: (-)-3-methoxy-N-methylmorphinan
الأسماء التجارية: (Ro.1-5470/6)
ملاحظة: ديكستروميثورفان (Dextromethorphan) لا يعتبر مادة مخدرة.
---
 81. مثيل ثنائي هيدرومورفين (Methyldihydromorphine)
الاسم الكيميائي: 6-مثيل ثنائي هيدرومورفين 
Chemical Name: 6-methyldihydromorphine
الأسماء التجارية: 2178
---
 82. مثيل ديزورفين (Methyldesorphine)
الاسم الكيميائي: 6-مثيل-دلتا-6-دي أوكسي مورفين 
Chemical Name: 6-methyl-delta-6-deoxymorphine
الأسماء التجارية: Methyldesomorphin(MK57)
---
 83. مستخلصات قش الخشخاش (Concentrate of poppy straw)
الوصف: 
المادة الناتجة من عملية تركيز قلويات قش الخشخاش.
Chemical Name: The material arising when poppy straw has entered into a process for the concentration of its alkaloids when such material is made available in trade.
---
 84. وسيط الموراميد (Moramide Intermediate)
الاسم الكيميائي: 2-مثيل-3-مورفولينو-1،1-ثنائي فنيل بروبان حمض كاربوكسيليك 
Chemical Name: 2-methyl-3-morpholino-1,1-diphenylpropane carboxylic acid / 1,1-diphenyl-2-methyl-3-morpholino propanecarboxylic acid
الأسماء التجارية: Pre-moramide
---
 85. مورفيريدين (Morpheridine)
الاسم الكيميائي: 1-(2-مورفولينوإثيل)-4-فنيل بيبريدين-4-حمض كاربوكسيليك استر إثيلي 
Chemical Name: 1-(2-morpholinoethyl)-4-phenylpiperidine-4-carboxylic acid ethyl ester
الأسماء التجارية: Morpholino-ethylnorpethidine
---
 86. مورفين (Morphine)
الاسم الكيميائي: 7،8-ديهيدرو-4،5-إيبوكسي-3،6-ثنائي هيدروكسي-ن-مثيل مورفينان 
Chemical Name: 7,8-dehydro-4,5-epoxy-3,6-dihydroxy-N-methyl-morphinan
الوصف: 
كافة مستحضرات المورفين المدرجة وغير المدرجة في دساتير الأدوية والتي تحتوي على أكثر من 0.2% من المورفين، مخففات المورفين في مادة غير فعالة سائلة أو صلبة أيًا كانت درجة تركيزها.
---
 87. ميتازوسين (Metazocine)
الاسم الكيميائي: 2-هيدروكسي-2،5،9-ثلاثي مثيل-6،7-بنزومورفان 
Chemical Name: 2-hydroxy-2,5,9-trimethyl-6,7-benzomorphan / 1,2,3,4,5,6-hexahydro-8-hydroxy-3,6,11-trimethyl-2,6-methano-3-benzazocine
الأسماء التجارية: Methobenzorphan(N.I.H.7410)
---
 88. ميتوبون (Metopon)
الاسم الكيميائي: 5-مثيل ثنائي هيدرومورفينون 
Chemical Name: 5-methyldihydromorphinone
الأسماء التجارية: Methyldihydromorphinone
---
 89. ميثادون (Methadone)
الاسم الكيميائي: 6-ثنائي مثيل أمينو-4،4-ثنائي فنيل-3-هيبتانون 
Chemical Name: 6-dimethylamino-4,4-diphenyl-3-heptanone
الأسماء التجارية: Amilone, Heptanone, Polamidon, Dolophin, Physeptone
---
 90. وسيط الميثادون (Methadone Intermediate)
الاسم الكيميائي: 4-سيانو-2-ثنائي مثيل أمينو-4،4-ثنائي فنيل بيوتان 
Chemical Name: 4-cyano-2-dimethylamino-4,4-diphenyl butane / 2-dimethylamino-4,4-diphenyl-4-cyanobutane
الأسماء التجارية: Pre-methadone
---
 91. ميثامفيتامين (Methamphetamine)
الاسم الكيميائي: (+)-2 مثيل أمينو-1-فنيل بروبان 
Chemical Name: (+)-2-methylamino-1-phenylpropane
الأسماء التجارية: Methedrine
---
 92. ميثاكوالون (Methaqualone)
الاسم الكيميائي: 2-مثيل-3-أورثو-توليل-4(3H)-كينازولينون 
Chemical Name: 2-methyl-3-O-tolyl-4-(3H)-quinazolinone
الأسماء التجارية: Revonal
---
 93. مثيل فينيدات (Methylphenidate)
الاسم الكيميائي: 2-فنيل-2-(2-بيبريديل) استر مثيل حمض الخليك 
Chemical Name: 2-phenyl-2-(2-piperidyl) acetic acid methyl ester
الوصف: بذاته وأملاحه بذاتها في جميع أشكالها الصيدلية المختلفة.
الأسماء التجارية: Ritalin
---
 94. ميروفين (Myrophine)
الاسم الكيميائي: ميريستيل بنزيل مورفين 
Chemical Name: Myristylbenzylmorphine
الأسماء التجارية: Myristyl peronine-(N.I.H.5986A.)
---
 95. نورا أسيميثادول (Noracymethadol)
الاسم الكيميائي: (±)-ألفا-3-أسيتوكسي-6-مثيل أمينو-4،4-ثنائي فنيل هيبتان 
Chemical Name: (±)-α-3-acetoxy-6-methylamino-4,4-diphenylheptane
الأسماء التجارية: (N.I.H.-7667)
---
 96. نوربيبانون (Norpipanone)
الاسم الكيميائي: 4،4-ثنائي فنيل-6-بيبريدينو-3-هيكسانون 
Chemical Name: 4,4-diphenyl-6-piperidino-3-hexanone
الأسماء التجارية: Hexalgon
---
 97. نورليفورفانول (Norlevorphanol)
الاسم الكيميائي: (-)-3-هيدروكسي مورفينان 
Chemical Name: (-)-3-hydroxymorphinan
الأسماء التجارية: (Ro.-1-7687)(N.I.H.-7539)
---
 98. نورمورفين (Normorphine)
الاسم الكيميائي: دي مثيل مورفين 
Chemical Name: Demethylmorphine / N-demethylated morphine
---
 99. نورميثادون (Normethadone)
الاسم الكيميائي: 6-ثنائي مثيل أمينو-4،4-ثنائي فنيل-3-هيكسانون 
Chemical Name: 6-dimethylamino-4,4-diphenyl-3-hexanone / 1-dimethylamino-3,3-diphenyl-4-hexanone / 1,1-diphenyl-1-dimethylaminoethyl-2-butanone
الأسماء التجارية: Deatussan, Extussin, Mepidon, Veryl, Ticarda
---
 100. نيكومورفين (Nicomorphine)
الاسم الكيميائي: 3،6-ثنائي نيكوتينيل مورفين 
Chemical Name: 3,6-dinicotinylmorphine / Di-nicotinic acid ester of morphine
الأسماء التجارية: Nicophine, Vendal
---
 101. تتراهيدروكانابينول (Tetrahydrocannabinol)
الاسم الكيميائي: 1-هيدروكسي-3-بنتيل-6a،7،10،10a-رباعي هيدرو-6،6،9-ثلاثي مثيل-6H-ثنائي بنزو(b،d)بيران 
Chemical Name: 1-Hydroxy-3-pentyl-6a,7,10,10a-tetrahydro-6,6,9-trimethyl-6H-dibenzo(b,d)pyran
---
 102. إس تي بي، دي أو إم (STP, DOM)
الاسم الكيميائي: 2-أمينو-1-(2،5-ثنائي ميثوكسي-4-مثيل) فنيل بروبان 
Chemical Name: 2-amino-1-(2,5-dimethoxy-4-methyl)phenylpropane
---
 103. دي إم إتش بي (DMHP)
الاسم الكيميائي: 3-(1،2 ثنائي مثيل هيبتيل)-1-هيدروكسي-7،8،9،10-رباعي هيدرو-6،6،9 ثلاثي مثيل-6H-ثنائي بنزو(b،d)بيران 
Chemical Name: 3-(1,2-dimethylheptyl)-1-hydroxy-7,8,9,10-tetrahydro-6,6,9-trimethyl-6H-dibenzo(b,d)pyran
---
 104. سيلوسين وسيلوتسين (Psilocine, Psilotsin)
الاسم الكيميائي: 3-(2-ثنائي مثيل أمينو إثيل)-4-هيدروكسي إندول 
Chemical Name: 3-(2-dimethylaminoethyl)-4-hydroxyindole
---
 105. مسكالين (Mescaline)
الاسم الكيميائي: 3،4،5 ثلاثي ميثوكسي فين إثيل أمين 
Chemical Name: 3,4,5-trimethoxyphenethylamine
---
 106. باراهيكسيل (Parahexyl)
الاسم الكيميائي: 3-هيكسيل-1-هيدروكسي 7،8،9،10-رباعي هيدرو-6،6،9 ثلاثي مثيل-6H-ثنائي بنزو(b،d)بيران 
Chemical Name: 3-hexyl-1-hydroxy-7,8,9,10-tetrahydro-6,6,9-trimethyl-6H-dibenzo(b,d)pyran
---
 107. دي إي تي (DET)
الاسم الكيميائي: ن،ن ثنائي إثيل تريبتامين 
Chemical Name: N,N-diethyltryptamine
---
 108. دي إم تي (DMT)
الاسم الكيميائي: ن،ن ثنائي مثيل تريبتامين 
Chemical Name: N,N-dimethyltryptamine
---
 109. ميكلوكوالون (Mecloqualone)
الاسم الكيميائي: 3-(أورثو-كلورفنيل)-2-مثيل-4-(3H)-كينازولينون 
Chemical Name: 3-(O-Chlorophenyl)-2-methyl-4-(3H)-quinazolinone
---
 110. تينوسيكليدين (Tenocyclidine)
الاسم الكيميائي: 1-[1-(2-ثينيل) سيكلوهيكسيل] بيبريدين 
Chemical Name: 1-[1-(2-thienyl)cyclohexyl]piperidine
الأسماء التجارية: TCP
---
 111. روليسيكليدين (Rolicyclidine)
الاسم الكيميائي: 1-(1-فنيل سيكلوهيكسيل) بيروليدين 
Chemical Name: 1-(1-phenylcyclohexyl)pyrrolidine
الأسماء التجارية: PHF or PCPY
---
 112. إتيسيكليدين (Eticyclidine)
الاسم الكيميائي: ن-إثيل-1-فنيل سيكلوهيكسيل أمين 
Chemical Name: N-ethyl-1-phenyl cyclohexylamine
الأسماء التجارية: PCE
---
 113. بنزفيتامين (Benzfetamine)
الاسم الكيميائي: ن-بنزيل-ن-ألفا-ثنائي مثيل فين إثيل أمين 
Chemical Name: N-benzyl-N-α-dimethylphenethylamine
الوصف: بذاتها وأملاحها بذاتها في جميع أشكالها الصيدلية المختلفة.
---
 114. ألفنتانيل (Alfentanil)
الاسم الكيميائي: ن-[1-[2-(4-إثيل-4،5-ثنائي هيدرو-5-أوكسو-1H-تترازول-1-يل) إثيل]-4-(ميثوكسي مثيل)-4-بيريدينيل]-ن-فنيل بروباناميد 
Chemical Name: N-{1-{2-(4-ethyl-4,5-dihydro-5-oxo-1H-tetrazol-1-yl)ethyl}-4-(methoxymethyl)-4-piperidinyl}-N-phenylpropanamide
الأسماء التجارية: Rapifen
---
 115. برولامفيتامين (Brolamfetamine - DOB)
الاسم الكيميائي: (±)-4-برومو-2،5-ثنائي ميثوكسي-ألفا-مثيل فين إثيل أمين 
Chemical Name: (±)-4-bromo-2,5-dimethoxy-α-methylphenethylamine / 2,5-dimethoxy-4-bromoamphetamine
الأسماء البديلة: داي ميثوكس برمو امفيتامين (Dimethoxybromoamfetamine)
---
 116. تينامفيتامين (Tenamfetamine - MDA)
الاسم الكيميائي: ألفا-مثيل-3،4-(مثيلين ثنائي أوكسي) فين إثيل أمين 
Chemical Name: α-methyl-3,4(methylenedioxy)phenethylamine
الأسماء البديلة: ميثيلين ثنائي أوكسي امفيتامين (Methylenedioxyamphetamine)
---
 117. بنتازوسين (Pentazocine)
الاسم الكيميائي: 1،2،3،4،5،6-سداسي هيدرو-6،11-ثنائي مثيل-3-(3-مثيل-2 بيوتنيل)-2،6 ميثانو-3-بنزازوسين-8-أول 
Chemical Name: 1,2,3,4,5,6-hexahydro-6,11-dimethyl-3-(3-methyl-2-butenyl)-2,6-methano-3-benzazocin-8-ol
الأسماء التجارية: Sosegon, Fortral, Talwin
---
 118. سوفنتانيل (Sufentanil)
الاسم الكيميائي: ن-[4-(ميثوكسي مثيل)-1-{2-(2-ثينيل)-إثيل}-4-بيبريديل] بروبيونانيليد 
Chemical Name: N-{4-(methoxymethyl)-1-{2-(2-Thienyl)-ethyl}-4-piperidyl}propionanilide
---
 119. ثيوفنتانيل (Thiofentanyl)
الاسم الكيميائي: ن-[1-[2-(2-ثينيل) إثيل]-4-بيبريديل] بروبيونانيليد 
Chemical Name: N-{1-[2-(2-thienyl)ethyl]-4-piperidyl}propionanilide
---
 120. فنيتيلين (Fenetyline)
الاسم الكيميائي: 7-[2-(ألفا-مثيل فين إثيل) أمينو] إثيل] ثيوفيللين 
Chemical Name: 7-{2-{(α-methylphenethyl)amino}ethyl}theophylline
---
 121. ألفا مثيل فنتانيل (Alpha-methylfentanyl)
الاسم الكيميائي: ن-[1-(ألفا-مثيل فين إثيل)-4-بيبريديل] بروبيونانيليد 
Chemical Name: N-{1-(α-methylphenethyl)-4-piperidyl}propionanilide
---
 122. بارا-فلورفنتانيل (Para-fluorofentanyl)
الاسم الكيميائي: 4-فلورو-ن-(1-فين إثيل-4-بيبريديل) بروبيونانيليد 
Chemical Name: 4-fluoro-N-(1-phenethyl-4-piperidyl)propionanilide
---
 123. بيتا-هيدروكسي فنتانيل (Beta-hydroxyfentanyl)
الاسم الكيميائي: ن-[1-(بيتا هيدروكسي فين إثيل)-4-بيبريديل] بروبيونانيليد 
Chemical Name: N-{1-(beta-hydroxy phenethyl)-4-piperidyl}propionanilide
---
 124. بيتا-هيدروكسي-3-مثيل فنتانيل (Beta-hydroxy-3-methylfentanyl)
الاسم الكيميائي: ن-[1-(بيتا هيدروكسي فين إثيل)-3-مثيل-4-بيبريديل] بروبيونانيليد 
Chemical Name: N-{1-(beta-hydroxy phenethyl)-3-methyl-4-piperidyl}propionanilide
---
 125. 3-مثيل فنتانيل (3-Methylfentanyl)
الاسم الكيميائي: ن-(3-مثيل-1-فين إثيل-4-بيبريديل) بروبيونانيليد 
Chemical Name: N-(3-methyl-1-phenethyl-4-piperidyl)propionanilide
---
 126. كاثينون (Cathinone)
الاسم الكيميائي: (-)-ألفا-إمينو بروبيوفينون 
Chemical Name: (-)-alpha-aminopropiophenone / (-)(S)-2-aminopropiophenone
---
 127. ميثاكاثينون (Methcathinone)
الاسم الكيميائي: 2-(مثيل أمينو)-1-فنيل بروبان-1-أون 
Chemical Name: 2-(methylamino)-1-phenylpropan-1-one
الأسماء التجارية: Ephedrone (افيدرون)
---
 128. إرتيبتامين (Ertryptamine)
الاسم الكيميائي: 3-(2-أمينو بوتيل) إندول 
Chemical Name: 3-(2-aminobutyl)indole
---
 129. أمينوركس (Aminorex)
الاسم الكيميائي: 2-أمينو-5-فينيل-2-أوكسازولين 
Chemical Name: 2-amino-5-phenyl-2-oxazoline
---
 130. 4-مثيل أمينوركس (4-Methyl aminorex)
الاسم الكيميائي: (±) مقرون-2-أمينو-4-مثيل-5-فنيل-2-أوكسازولين 
Chemical Name: (±) cis-2-amino-4-methyl-5-phenyl-2-oxazoline
---
 البنود الختامية
ملاحظة عامة: 
وكذلك أي مستحضر أو مخلوط أو مستخلص أو أي مركب آخر يحتوي على إحدى المواد المدرجة في هذا الجدول أو أي أحد أملاحها أو نظائرها أو استيراتها أو إثيراتها أو أملاح النظائر والإسترات والإثيرات لهذه المواد وبأي نسبة كانت ما لم ينص على نسبة محددة.
___________________________________________
مواد إضافية مدرجة
مادة الفلونيترازيبام (Flunitrazepam) ومستحضراتها
الاسم الكيميائي: 5-(أورثو-فلوروفينيل)-1،3-داي-هيدرو-1-ميثيل-7 نترو-2H-1،4-بنزودايازيبين-2-أون 
Chemical Name: 5-(O-Fluorophenyl)-1,3-Dihydro-1-Methyl-7-Nitro-2H-1,4-Benzodiazepine-2-One
---
 (أ) داي هيدرو إتروفين (Dihydroetorphine)
الاسم الكيميائي: 7،8-ثنائي هيدرو-7 ألفا-[1-(أر)-هيدروكسي-1-مثيل بيوتيل]-6،14-إندوإيثانوتتراهيدروأوريبافين 
Chemical Name: 7,8-Dihydro-7-α-[(R)-Hydroxy-1-Methylbutyl]-6,14-Endo Ethanotetrahydrooripavine
---
 (ب) ريمفنتانيل (Remifentanil)
الاسم الكيميائي: 1-(2-ميثوكسى كاربونيل-إيثيل)-4-(فنيل بروبيونيل أمينو بيبريدين-4-كاربوكسيليك أسيد مثيل استر 
Chemical Name: 1-(2-Methoxycarbonyl-Ethyl)-4-(Phenylpropionylamino)-Piperidine-4-Carboxylic Acid Methyl ester
---
 (ج) إيسوميرات (Isomers)
جميع المواد المدرجة بالجدول الأول.
---
 (د) إسترات وإيثرات (Ethers and Esters)
جميع المواد المدرجة بالجدول الأول.
---
 (هـ) أملاح
جميع المواد المدرجة بالجدول الأول بما فيها أملاح الإسترات والإثيرات الإيسوميرات في حالة وجود هذه الأملاح.
---
 (و) ستيروإيسوميرات (Stereoisomers)
جميع المواد المدرجة بالجدول الأول.
---
 مركبات امفيتامين ومشتقاتها
 1. دي إم إيه (DMA)
الاسم الكيميائي: (±)-2،5-ثنائي ميثوكسي-ألفا-ميثيل فين إثيل أمين 
Chemical Name: (±)-2,5-dimethoxy-α-methylphenethylamine
---
 2. إم دي إم إيه (MDMA)
الاسم الكيميائي: (±) ن، ألفا-ثنائي ميثيل-3،4-(ميثيلين-ثنائي أوكسي) فين إيثيل أمين 
Chemical Name: (±) N,α-dimethyl-3,4(methylene-dioxy)phenethylamine
---
 3. إم إم دي إيه (MMDA)
الاسم الكيميائي: 2-ميثوكسي-ألفا-ميثيل-4،5-(ميثيلين ثنائي أوكسي) فين إثيل أمين 
Chemical Name: 2-methoxy-α-methyl-4,5-(methylenedioxy)phenethylamine
---
 4. ن-إثيل إم دي إيه (N-ethyl MDA)
الاسم الكيميائي: (±)-ن-إثيل-ألفا-ميثيل-3،4 (ميثيلين ثنائي أوكسي) فين إثيل أمين 
Chemical Name: (±)-N-ethyl-α-methyl 3,4-(methylenedioxy)phenethylamine
---
 5. ن-هيدروكسي إم دي إيه (N-hydroxy MDA)
الاسم الكيميائي: (±)-ن-(ألفا-ميثيل-3،4-(ميثيلين ثنائي أوكسي) فين إثيل) هيدروكسيل أمين 
Chemical Name: (±)-N-(α-methyl-3,4-methylenedioxy phenethyl)hydroxylamine
---
 6. بي إم إيه (PMA)
الاسم الكيميائي: بارا-ميثوكسي-ألفا-ميثيل فين-إثيل أمين 
Chemical Name: p-methoxy-α-methylphenethylamine
---
 7. ليفامفيتامين (Levamphetamine)
الاسم الكيميائي: (-) (أر)-ألفا-ميثيل فين إثيل أمين 
Chemical Name: (-)(R)-α-methylphenethylamine
---
 8. ليفوميثامفيتامين (Levomethamphetamine)
الاسم الكيميائي: (-)-ن، ألفا-ثنائي ميثيل فين إثيل أمين 
Chemical Name: (-)-N,α-dimethylphenethylamine
---
 9. تي إم إيه (TMA)
الاسم الكيميائي: (±)-3،4،5-ثلاثي ميثوكسي-ألفا-ميثيل فين إثيل أمين 
Chemical Name: (±)-3,4,5-trimethoxy-α-methylphenethylamine
---
 10. إيتيل أمفيتامين (Ethylamphetamine)
الاسم الكيميائي: ن-إثيل ألفا-ميثيل فين إثيل أمين 
Chemical Name: N-ethylamphetamine / N-ethyl-α-methyl phenethylamine
---
 11. دي أو إي تي (DOET)
الاسم الكيميائي: (±) 4-إثيل-2،5-ثنائي ميثوكسي ألفا فين إثيل أمين 
Chemical Name: (±) 4-ethyl-2,5-dimethoxy-α-phenethylamine
---
 2-سي بي (2-CB)
الاسم الكيميائي: 4-برومو-2،5-داي ميثوكسى فينيثيل أمين 
Chemical Name: 4-Bromo-2,5-dimethoxy Phenethylamine
---
 4-إم تي إيه (4-MTA)
الاسم الكيميائي: ألفا ميثيل-4-ميثيل ثيو فينيثيل أمين 
Chemical Name: α-Methyl-4-Methylthiophenethylamine
---
 مادة الترامادول (Tramadol)
وأملاحها ونظائرها واستراتها وإيثراتها وأملاح نظائرها واستراتها ومستحضراتها.
---
 مركبات القنب الصناعي (Synthetic Cannabinoids)
1. Pentyl-3-(1-naphthoyl)indole (JWH-018)
2. 1-butyl-3-(1-naphthoyl)indole (JWH-073)
3. 1-[2-(4-morpholinyl)ethyl]-3-(1-naphthoyl)indole (JWH-200)
4. 5-[1,1-dimethylheptyl)-2-(3-hydroxycyclohexyl)phenol (cannabicyclohexanol, CP-47,497 C8 homologue)
5. 1-AB-Fubinaca: (N-{(2S)-1 amino-3-methyl-1-oxobutan-2-yl}-1-{(4-fluorophenyl)methyl}indazole-3-carboxamide
6. 2-AB-Chminaca: N-{(2S)-1-amino-3-methyl-1-oxobutan-2-yl}-1-(cyclohexylmethyl)indazole-3-carboxamide
7. 3-XLR-11: (1-(5-fluoropentyl)-1H-indol-3-yl)(2,2,3,3-Tetramethylcyclopropyl)methanone
8. 4-XLR-11N-(4-fluoropentyl)-1 isomer
9. 5-FYB-AMB (AMB-Fubinaca)(MMB-FUBINCA): methyl(2S)-2-{1-({4-fluorophenyl)methyl}Indazole-3-carbonyl)amino}-3-methylbutanoate
10. 6-5-fluoro-ADB: methyl (S)-2-(1-(5-fluoropentyl)-1H-Indazole-3-carboxamido)-3,3-dimethylbutanoate
11. 1-SDB-005: naphthalene-1-yl 1-pentyl-1H-indazole-3-carboxylate
12. 2-EMB-Fubinaca: ETHYL {1-(4-Fluorobenzyl)-1H-indazole-3-Carbonyl}-L-Valinate
13. 3-5-FLUOROPY-PICA: {1-(5-fluoropentyl)-1H-Indol-3-YL}{Pyrrolidin-1-YL}METHANONE
14. 4-ETHCATHINONE: 2-ETHYLAMINO-1-PHENYL-PROPAN-1-ONE
15. 5-4-CHLOROETHCATHINONE: 1-(4-CHLOROPHENYL)-2-(ETHYLAMINO)PROPAN-1-ONE
---
 مجموعات كيميائية
- NAPHTHOYLINDOLES
- NAPHTHOYLMETHYLINDOLES
- NAPHTHOYLPYRROLES
- NAPHTHOYLMETHYLINDENES
- PHENYLACETYLINDOLES
- CYCLOHEXYLPHENOLS
- CLASSICAL CANNABINOIDS.DIBENZOPYRAN
---
 مشتقات الكاثينون (Substituted Cathinones)
1. Substituted cathinone:
 a) Methcathinone
 b) Methyl Ethcathinone (MEC)
 c) Phenthylamine Chemical Class
2. Pyrovaleron
3. Indazole Carboxamides
4. Tetramethyl Cyclopropyl Indoles
5. Adamantoyl Indoles
6. Quinolinyl Ester
---
 مشتقات الفنتانيل (Fentanyl Derivatives)
1. para-Fluorobutyryl fentanyl (p-FBF) 
 N-(4-fluorophenyl)-N-[1-(2-phenylethyl)-4-piperidinyl]-butanamide
2. ortho-Fluorofentanyl 
 N-(2-fluorophenyl)-N-[1-(2-phenylethyl)-4-piperidinyl]propanamide
3. Methoxyacetylfentanyl 
 2-Methoxy-N-(1-phenethylpiperidin-4-yl)-N-phenylacetamide
4. Cyclopropylfentanyl 
 N-(1-phenethylpiperidin-4-yl)-N-phenylcyclopropanecarboxamide
5. Carfentanil or carfentanyl 
 - Methyl 1-(2-phenylethyl)-4-[phenyl(propanoyl)amino]piperidine-4-carboxylate
 - 4-((1-Oxopropyl)Phenylamino)-1-(2-phenylethyl)-4-piperidinecarboxylic acid methyl ester
6. Ocfentanil 
 N-(2-FluoroPhenyl)-2-methoxy-N-[1-(2-Phenylethyl)Piperidin-4-yl]acetamide
7. Furanylfentanyl (Fu-F) 
 N-Phenyl-N-[1-(2-phenylethyl)piperidin-4-yl]furan-2-carboxamide
8. Acryloylfentanyl (Acrylfentanyl) 
 N-Phenyl-N-[1-(2-phenylethyl)Piperidin-4-yl]prop-2-enamide
9. 4-Fluoroisobutyrfentanyl (4-FIBF) 
 N-(4-Fluorophenyl)-N-[1-(2-phenylethyl)-4-piperidinyl]-isobutanamide
10. Tetrahydrofuranylfentanyl (THF-F) 
 N-Phenyl-N-[1-(2-phenylethyl)piperidin-4-yl]oxolane-2-carboxamide
11. Crotonylfentanyl (isomer of cyclopropylfentanyl) 
 (2E)-N-Phenyl-N-[1-(2-phenylethyl)-4-piperidinyl]-2-butenamide
12. Valeryl Fentanyl (Pentanyl fentanyl, Pentanoyl fentanyl) 
 N-Phenyl-N-[1-(2-Phenylethyl)-4-piperidinyl]-pentanamide
---
 مواد أخرى
13. U-47700 (Pink Heroin) 
 Trans-3,4-dichloro-N-(2(dimethylamino)cyclohexyl)-N-methyl-benzamide
14. Ethylphenidate (EPH) 
 (RS)-Ethyl-2-phenyl-2-(piperidin-2-yl)acetate
15. Methiopropamine (MPA) 
 1-(thiophen-2-yl)-2-methylaminopropane
16. Tilidine 
 (Ethyl (1R,2S)-2-(dimethylamino)-1-phenylcyclohex-3-ENE-1-carboxylate)
17. Isotonitazene 
 N,N-diethyl-2-(isopropoxybenzyl)-5-nitro-1H-benzo[d]imidazole-1-yl)Ethan-1-amine
18. 3-Methoxyphencyclidine (3-MeO-PCP) 
 1-[1-(3-methoxyphenyl)cyclohexyl]piperidine
19. Diphenidine 
 1-(1,2-diphenylethyl)piperidine
20. Cumyl-Pegaclone 
 5-Pentyl-2-(2-phenylpropan-2-yl)-2,5-dihydro-1H-pyrido[4.3-b]indol-1-one
__________________________________________
 الجدول رقم (2)
المستحضرات المستثناة من النظام المطبق على المواد المخدرة:
(1) مستحضرات المورفين: للبوس الواحد
(1) لبوس يود فورم، يود فورم .......... 0.320 جرام.
والمورفين كلوريدات المورفين .......... 0.016 جرام
زبدة الكاكاو – كمية كافية لغاية جرام واحد.
(2) لصقة الأفيون:
راتنج لامي ................................. 20 جرام
تربنتينا ................................. 30 جرام
جمع أصفر .................................. 15 جرام
مسحوق لبان ذكر ............................ 18 جرام
مسحوق الجاوي ............................. 10 جرام
مسحوق الأفيون ................................ 5 جرام
بلسم الميرو .................................. 2 جرام
(3) لصقة الأفيون:
خلاصة الأفيون ............................ 25 جرام
راتنج لامي منقى ............................ 25 جرام
لصقة الرصاص الصمغية ........................... 50 جرام
(4) لصقة الأفيون:
راتنج لامي ................................. 8 جرام
تربنتينا عادة ................................ 15 جرام
جمع أصفر ................................. 5 جرام
لبان ذكر مسحوق ............................... 8 جرام
جاوى مسحوق ........................... 4 جرام
مسحوق الأفيون ............................. 2 جرام
بلسم الميرو ........................... 90 جرام
(5) لصقة الأفيون:
مسحوق الأفيون الناعم ............................. 10 جرام
لصقة راتنجية ............................... 90 جرام
(6) لصقة الأفيون:
(انظر التركيب تحت رقم 5)
مخلوط بغيرها من اللصقات الواردة بالفارماكوبيا البريطانية أو بكودكس الصيدلة البريطاني.
(7) مروخ الأفيون:
صبغة الأفيون ........................... 500 ملليمتر
مروخ صابوني ............................... 500 ملليمتر
(8) مروخ أفيون:
(انظر التركيب الوارد تحت رقم 7)
مخلوطا بأحد المروخات الواردة بالفارماكوبيا البريطانية أو بكودكس البريطاني.
(9) مروخ الأفيون النوشادري، مروخ الكافور النوشادري 30 ملليلتر
صبغة الأفيون .............................. 30 ملليلتر
مروخ البلادونا ................................ 5 ملليلتر
محلول النوشادر المركز ............................ 5 ملليلتر
مروخ صابوني كمية كافية لغاية ....................... 100 ملليلتر
(10) مروخ الأفيون النوشادري (نفس التركيب الوارد تحت رقم 9) مخلوطا بأحد المروخات الواردة بالفارماكوبيا البريطانية أو بكودكس الصيدلة البريطاني.
(11) عجائن كاوية للأعصاب ومستحضرات تحتوي – عدا أملاح المورفين أ أملاح المورفين والكوكايين على ما لا يقل عن 25% من الأحماض الزرنيخية، ويدخل في صنعها كربوزوت أو فينول بالمقدار اللازم لتكون متماسكة على شكل عجينة.
(12) حبوب مضادة للإسهال:
كافور ............................. 0.648 جرام
خلات الرصاص ........................ 0.013 جرام
تحت نترات البزموت ....................... 0.162 جرام
حمض الثنيك .......................... 0.648 جرام
مسحوق الأفيون .......................... 0.020 جرام
(13) حبوب الديجيتالا والأفيون المركبة:
مسحوق أوراق الديجيتالا ................... 0.031 جرام
مسحوق الأفيون ........................ 0.019 جرام
مسحوق عرق الذهب ......................... 0.013 جرام
كبريتات الكينين ........................... 0.078 جرام
شراب الجلوكوز لعمل 12 حبة ............ كمية كافية
(14) حبوب الزئبق:
مع الأفيون حبوب الزئبق ...................... 3.089 جرام
مسحوق الأفيون لعمل 12 حبة .................... 0.19 جرام
(15) حبوب الزئبق مع الطباشير والأفيون:
مسحوق عرق الذهب بالأفيون ....................... 0.78 جرام.
(تركيب هذا المسحوق مبين تحت رقم 21).
مسحوق الزئبق بالطباشير ................. 0.078 جرام
سكر لبن كمية كافية
شراب الجلوكوز كمية كافية لعمل 12 حبة.
(16) حبوب عرق الذهب مع بصل العنصل:
مسحوق عرق الذهب بالأفيون .................... 30 جرام
(تركيب هذا المسحوق مبين تحت رقم 21)
مسحوق بصل العنصل ........................ 10 جرام
راتنج نوشادري مسحوق ...................... 10 جرام
شراب الجلوكوز – كمية كافية
(17) حبوب كلور الزئبقيك بالأفيون:
كلورور الزئبقيك المسحوق ..................... 0.10 جرام
خلاصة الأفيون ............................. 0.20 جرام
خلاصة عرق النجيل ............................ 0.20 جرام
(مسحوق عرقسوس كمية كافية لعمل 10 حبات)
(18) حبوب يودور الزئبقوز يودور الزئبقوز الحديث
التحضير ............................ 0.50 جرام
مسحوق الأفيون .............................. 0.20 جرام
مسحوق عرقسوس ............................. 0.3 جرام
عسل أبيض – كمية كافية لعمل 10 حبات.
(19) حبوب الرصاص مع الأفيون:
خلات الرصاص المسحوق ...................... 80 جرام
مسحوق الأفيون ........................... 18 جرام
شراب الجلوكوز أو كمية كافية ............... 8 جرام
(20) حبوب التربنتينا المركبة: أفيون .......... 0.05 جرام
كبريتات الكينين ........................... 2.05 جرام
ميعة سائلة ............................. 3.00 جرام
تربنتينا ............................ 8.00 جرام
كربونات المغنزيوم .... كمية كافية لعمل مائة حبة.
(21) مسحوق عرق الذهب، مسحوق عرق الذهب
المركب ..................... 10.00 جرام
(مسحوق دوفر) مسحوق الأفيون ............. 10.00 جرام
مسحوق كبريتات البوتاسيوم ................... 80.00 جرام
(22) مخاليط مسحوق دوفر (انظر التركيب الوارد تحت رقم 21) مع الزئبق الطباشيري أو الأسبيرين أو الفيناستين أو الكينين وأملاحه أو بيكربونات الصودا.
(23) مسحوق الكينو المركب:
مسحوق الكينو ............................ 75 جرام
مسحوق الأفيون ........................... 5 جرام
مسحوق القرفة .............................. 20 جرام
(24) أقماع الرصاص المركبة:
خلات الرصاص المسحوقة ................... 2.4 جرام
مسحوق الأفيون .......................... 0.8 جرام
زبدة الكاكاو – كمية كافية لعمل 12 قمعا زنة كل منها حوالي جرام واحد.
(25) أقراص مضادة للزكام رقم 2:
مسحوق الأفيون ......................... 0.043 جرام
كبريتات الكينين ............................ 0.022 جرام
كلوريدات النوشادر ................................ 0.022 جرام
كافور ........................... 0.022 جرام
خلاصة أوراق البلادونا ............................ 0.043 جرام
خلاصة جذور خانق الذئب .......................... 0.043 جرام
(26) أقراص مضادة للإسهال رقم 2:
مسحوق الأفيون ........................... 0.016 جرام
كافور ............................. 0.016 جرام
مسحوق عرق الذهب ....................... 0.008 جرام
خلات الرصاص ...................... 0.011 جرام
(27) أقراص مضادة للدوسنطاريا:
مسحوق الأفيون ............................. 0.013 جرام
مسحوق عرق الذهب ...................... 0.648 جرام
مسحوق الزئبق الحلو ......................... 0.324 جرام
خلات الرصاص ................. 0.324 جرام
بزموت بيتانافاتول ....................... 0.1944 جرام
(28) أقراص الزئبق مع الأفيون:
كلور الزئبقوز المسحوق ................ 0.065 جرام
أكسيد الأنتيمون المسحوق .............. 0.065 جرام
مسحوق جذور عرق الذهب ................. 0.065 جرام
مسحوق الأفيون ....................... 0.065 جرام
سكر لبن ........................ 0.065 جرام
محلول الجيلاتين – كمية كافية لعمل قرص واحد.
(29) أقراص الرصاص مع الأفيون:
مسحوق خلات الرصاص الناعم ............. 19.44 جرام
مسحوق الأفيون ..................... 3.24 جرام
سكر مكرر مسحوق ...................... 6.48 جرام
محلول الثيوبرومين الأثيري ........... 3.60 ملليلتر
كحول .......................... 0.90 ملليلتر
(30) أقراص الرصاص مع الأفيون:
سكر الرصاص ..................... 0.195 جرام
مسحوق الأفيون .................... 0.060 جرام
محلول الجيلاتين كمية كافية لعمل قرص واحد:
(31) مرهم العفص المركب:
مسحوق العفص الناعم ....................... 20 جرام
خلاصة الأفيون ...................... 4 جرام
ماء مقطر ........................ 16 جرام
لانولين .......................... 10 جرام
برافين أصفر رخو ......................... 50 جرام
(32) مرهم العفص المركب:
(انظر التركيب الوارد تحت رقم 31 المخلوط بغيره من المراهم واللصقات الواردة بالفارماكوبيا البريطانية أو بكودكس الصيدلية البريطانية).
(33) مرهم العفص مع الأفيون:
مرهم العفص ......................... 2.005 جرام
مسحوق الأفيون ........................... 7.075 جرام
(34) مرهم العفص مع الأفيون:
(انظر التركيب الوارد تحت رقم 33 المخلوط بغيره من المراهم واللصقات الواردة بالفارماكوبيا البريطانية أو بكودكس الصيدلة) البريطاني.
(35) ياترين – 105 (حامض يودو أو كسيكينولاييك سلفونيك) مضافاً إليه 5% أفيون.
(ب) مستحضرات الديكوديد
محاليل الكارديازول ديكوديد:
محلول يحتوي على ما لا يقل عن 10% من الكاريديازول وما لا يزيد على 0.5% من أحد أملاح الديكوديد.
(ج) مستحضرات تلايكودال
(1) أقراص مضادة للأفيون:
أيكودال ........................... 1 جرام
مسحوق جنطيانا ............................. 35 جرام
مسحوق عرق الذهب ........................ 20 جرام
مبربيتات الكايتين ........................ 20 جرام
كافيين ............................. 5 جرام
سكر لبن ............................... 25 جرام
ملاحظة: يحظر عرض هذا المستحضر على الجمهور باسم مستحضر مضاد للأفيون.
(2) أقراص ب. ب المركبة:
مسحوق برياريس عادي ........................ 0.0324 جرام
جوز مقيء ........................... 0.0013 جرام
أيكودال .......................... 0.032 جرام
عرق الذهب .............................. 0.0648 جرام
راوند ....................... 0.0013 جرام
مسحوق القرفة المركب ................. 0.0324 جرام
طباشير عطري ........................ 0.0032 جرام
(د) مستحضرات الكوكايين:
1- حقن برناتزيك:
(أ) بي سيانور الزئبق .......... 0.03 جرام
كوكايين ....................... 0.02 جرام
(ب) سكيناميد الزئبق .................... 0.03 جرام
كوكايين ..................... 0.01 جرام
(2) حقن ستيلا:
(أ) سكسيناميد الزئبق ................ 0.03 جرام
كلوريدات الكوكايين ................... 0.01 جرام
(ب) سكسيناميد الزئبق .............. 0.05 جرام
كلوريدات الكوكايين ................... 0.03 جرام
(3) بي بورات الصودا المركب مع الكوكايين:
على شكل أقراص صلبة تحتوي على الأكثر على 0.2% من أحد أملاح الكوكايين مع لا يقل عن 20% من الأنتييرين أو من غيرها من المواد المسكنة المماثلة وما لا يزيد عن 40% من المواد المحسنة للطعم ولا يزيد وزن القرص عن جرام واحد.
(4) عجائن كاوية للأعصاب:
مستحضرات تحتوي – عدا أملاح الكوكايين أو أملاح الكوكايين
والمورفين على ما لا يقل عن 25% من الأحماض الزرنيخية ويدخل في صنعها كربوزوت أوفينول بالمقدار اللازم لتكون متماسكة على شكل عجينة.
(5) أقراص كوكايين وأتروبين تحتوي كل منها على 0.0003 جرام من أحد أملاح الكوكايين على الأكثر وعلى 0.0003 جرام من أحد أملاح الأتروبين على الأقل.
كبريتات الأتروبين .................... 0.0003 جرام
كلوريدات الكوكايين .................. 0.0003 جرام
سكر المن ............. 0.0003 جرام
زنة القرص الواحد .................... 0.0036 جرام
ونسبة الكوكايين فيه 8.3 %
(6) أقراص للصوت: كلوريدات البوتاس:
بورق.
كوكايين ................. 0.00025 جرام
زنة القرص الواحد ................... 0.335 جرام
(هـ) مستحضرات قاعدتها خلاصة أو صبغة القنب:
المستحضرات التي قاعدتها خلاصة أو صبغة القنب التي لا تستعمل إلا من الظاهر.
___________________________________________
 الجدول رقم (3)
في المواد التي تخضع لبعض قيود الجواهر المخدرة
(أ) المواد الآتية وكذلك مستحضراتها التى تحتوى على أى مادة من هذه المواد بكمية تزيد عن 100 ملليجرام فى الجرعة الواحدة ويتجاوز تركيزها فى المستحضر الواحد عن 2.5% ما لم ينص على غير ذلك.
1 – أثيل مورفين:
Ethyl morphin:
3 – أثيل مورفين
3- Ethyl morphine
مثل:
Dionine
2 – استيل ثنائى أيدروكودايين:
Acetyl dihydrocodeine:
6 – أسيتوكسى – 3 – ميثوكسى – ن – مثيل – 4 و5 أبوكسى – مورفينان.
6- acetoxy-3- methoxy-N- methyl-4,- epoxy- morphinan
مثل:
Acetylcodone.
6 – أيدروكسى – 3 – ميثوكسى – ن – مثيل – 4 و5 – ابوكسى – مورفينان.
6- hedroxy-3- methoxy-N- methyl-4,5- epoxy- morphian
مثل:
Dihydrin – Paracodin.
مورفولنييل اثيل مورفين.
Morpholinylethyl morphine.
أو:
بيتا – 4 – مورفولينيل أثيل مورفين
Beta-4- morpholinylethyl morphine
مثل:
Necodin.
3 – مثيل مورفين
3- Methylmorphine
مثل:
Methyl morphine
6 – نوركودايين:
Norcodeine:
ن – ديمثيل كودايين
N – Demethyl codeine
7 – نيكو ثنائى كواديين:
Nicodicodine:
6 – نيكوتنيل ثنائى أيدروكودايين
6- Nicotinyldihyrododeine
أو:
أستر حمض النيكوتنيك لثنائى أيدروكودايين
Nicotinic acid ester of dihydrocodeine
مثل:
N.I.H.8238-RC174.
(ب) المادة الآتية ومستحضراتها التى تحتوى على أكثر من 100 مللجرام بالجرعة الواحدة مع ما يساويها على الأقل من مادة المثيل سليولوز ما لم ينص على غير ذلك.
– بروبيرام:
Propiram:
ن – (1 – مثيل – 2 – بيبريدنواثيل) – ن – 2 – بيبريديل بروبيوناميد
N-(1- methyl-2- piperidinoethyl)-N-2- pyridylpropionamide
مثل:
Algeril.
(جـ) كذلك المواد الآتية:
1 – 1 – اثيل – 2 – كلوروفنيل اثنيل – كاربينول.
1-1- Ethyl-2- chlorovinylethinyl- carbinol
والمعروف بالأسم التجارى أو الأسم الدارج.
Ethchlorvynol
2 – اثينامات:
Ethinamate
1 – اثنيل سيكلوهيكسانول كاربامات
Ethinamate
2 – (ثنائى اثيل امينو) بروبيوفينون
2- Propiophenone (diethylamino)
4 – باربيتال
Barbital
5 و5 – ثنائى اثيل حمض باربتيوريك.
5,5- diethyl barbituric acid
5 – بنتوباربيتال
Pentobarbital
5 – اثيل – 5 – (1 – مثيل بيوتيل) حمض باربتيوريك
5- ethyl-5-5(1- methyl butyl) barbituric acid
6 – بيبرادول:
Pipradol
1و1 – ثنائى فنيل – 1 – (2 – بيبريديل) ميثانول.
1,1- diphenyl-1-(2- piperidyl) methanol
7 – (-) – 1 – ثنائى مثيل أمينو – 1 و2 – ثنائى فنيل ايثين.
(-)-1- dimethylamino-1,2- diphenylethane
والمعروف بالاسم التجارى أو الاسم الدارج بـ
S. P. A.
8 – سيكلوباربيتال:
Cyclobarbital
5 – 5 (1 – سيكلوهيكسامين – 1 – يل) – 5 – اثيل حمض باربيتيوريك.
5-5(1- cyclohexene-1-yl)-5- ethylbabituric acid
9 – فينسايكلدين:
Pheneyclidine
1 – (1 – فنيل سيكلوهيكسيل) بيبريدين.
1-(1- phenylcyclo hexyl) piperidine
10- فينمترازين:
Phenmetrazine
3 – مثيل – 2 – فنيل مورفولين.
3- methyl-2- phenylmorpholine
11 – فينو باربيتال:
Phenobarbital
5 – أثيل – 5 – فنيل حمض باربتيرريك
5- ethyl-5- phenyl barbituric acid
12 – مبروبامات:
Meprobamate
2 – مثيل بروبيل – 1 و3 – 1 و3 – بروبانيديول ثنائى كاربامات.
2- methyl- propyl-1,3 propanidiol dicarbamate
13 – مثيل فينو باربيتال:
Methyl Phenobarbital
5 – أثيل – 1 – مثيل – 5 – فنيل حمض باريتيوريك.
5- ethyl-1- methyl-5- phenyl barbituric acid
14 – مثييريلون:
Methyprylon
3 و3 – ثنائى أثيل – 5 – مثيل – 2 و4 – بييزيدين – ديون.
3,3- diethyl-5- methyl-2,4- piperidine- dion
15 – نيكوكودين:
Nicocodeine
6 – نيكوتنيل كودايين
6- Nicotinyl codeine
أو:
6 – (بيريدين – 3 – حمض كاربوكسليك) – كواديين أستز.
6-( pyridine-3- carboxylic acid)- codeine ester
وكذلك أملاح ونظائر واسترات واثيرات وأملاح نظائر واستيرات جميع المواد المذكورة فى هذا الجدول ما لم ينص على غير ذلك.
(أ) مادة: (4)-3,4- Dime Thyl-2- Phenylmobpholine
والمعروفة بالاسم الدولي غير التجاري Phendimetrazine
(ب) مادة: a-a-Dimethylphenethylamine
والمعروفة بالاسم الدولي غير التجاري Phentermine
(جـ) مادة: 5-(P-chlorphenyl)-2,5-Dihydro-3Himi(azol)
والمعروفة بالاسم الدولي غير التجاري Isoindol-5-olmazindol
الأفدرين وأملاحها
البيمولين، بوبرينورفين.
1- ن ــ حمض استيل الأنترانيل n-Acetylanthranilic acid
2- شبيه الإيفيدرين Pseudo ephedrine
3- الإيرجومترين Ergometrine
4- الإيرجوتامين Ergotamine
5- السافرول Safrol
6- الإيزوسافرول Isosafrol
7- 1- فنيل 2ـ بروبانون 1-phenyl-2-propanone
8- 3,4، مثيلين ديوكس فنيل – 2 – بروبانون
3,4-Methylenedioxy phenyl-2- Propanone
9- حمض الليسيرجيك Lysergic acid
10- بيبرونال Piperonal
11- ميزوكارب Mesocarb
12- زبيرول Ziperol
13- كاثين Cathine
14- اندريد الخليك Acetic anhydride
1- نورإفدرين:
(أر، أس) – ألفا – (1- أمينوإيثيل) بنزين ميثانول
1- Nor ephedrine:
(R,s)-a-(1- Amino Ethyl) Benzenemethano1
2- الوباربيتال
5,5 – ثنائي الليل حمض باربيتيوريك
2- Allobarbital
5,5- diallylbarbituric acid
3- بيوتوباربيتال:
5- بيوتيل – 5- أثيل حمض باربيتيوريك.
3- Butobarbital
-5- buty1-5- ethylbarbituric acid
1-4- Anilino-N-Phenethylpiperdine (ANPP)
2-3, 4- MDP-2- P methyl glycidate (PMK glycidate)
3-3, 4- MDP-2-P methyl glycidic acid
4-N- Phenethyl-4- piperidone (NPP)
5- Phenylacetic acid
6- alpha-Phenylacetoacetamide (APAA)
7- Alpha-Phenylacetoacetonitrile (APPAAN)
8- Methyl alpha-Phenylacetoacetate (MAPA)
4- أمينوبيريدين
4-aminopyridine 4-AP
1- بي أو سي – 4 – إي بي
Tert- buty14 – amilinopiperidine – 1carboxylate 1-boc-4-Ap
نورفينتانيل
N-phenyl-N-piperidin-4-y1propanamide Norfentany1
جى اتش بي (GHB)
جاما – حامض هيدروكسى البيوتيرات
g-Hydroxbutyric acid
(د) المواد الآتية وكذلك مستحضراتها المختلفة
1- مادة أمفيبرامون Amphepramon
-2- (ثنائي أيثيل أمينو) بروبيوفينون
-2- Diethylamino propiphenone
مشتقات البنزودبازبنيز Benzodiazepines ومستحضراتها
مادة كيتامين وأملاحها Ketamine ومستحضراتها
زولبيديم zolpidem
ن، ن6 ـ تراى ميثيل ـ 2 ـ بي ـ توليل اميدازو (1، 2 ـ ألفا) بيريدين ـ 3 ـ اسيتاميد.
N,N,6- Trimethyl-2-P- Tolylimidazo[1,2-a] Pyridine-3- Acetamide
ثنائي إيدروكدايين DIHYDROCODEINE
فولكودين PHOLCODINE
كودايين CODEINE
دكستروبروبكسيفين DEXTROPROPOXYPHENE
نالبوفين NALBUPHINE
تابنتادول TAPENTADOL
زاليبلون ZALEPLON
مادة البريجبالين وأملاحها وايثراتها ونظائرها وأملاح نظائرها واستراتها ومستحضراتها
1- Etizolam:
4-(2- chlorophenyl)-2- ethyl-9- methyl-6H- thieno [3,2-f] [1,2,4] triazolo [4,3-a] [1,4] diazepine
___________________________________________
الجدول رقم (4)
الحد الأقصى لكميات الجواهر المخدرة الذي لا يجوز للأطباء البشريين وأطباء الأسنان الحائزين على دبلوم أو بكالوريس تجاوزه في وصفة طبية واحدة
(1) الأفيون ................... 0.60 جرام
(2) المورفين وكافة أملاحه .................. 0.06 جرام
(أ) أقراص المورفين أو أملاحهاMorphine 420 ملليجرام (أربعمائة وعشرون ملليجرام).
(ب) أمبولات المورفين أو أملاحها Morphine 60 ملليجرام (ستون ملليجرام).
(3) داي أستيل المورفين (أسيتو مورفين ديامورفين)
(4) بنزويل المورفين وأملاحه وكافة استرات المورفين الأخرى وأملاحه .................... 0.02 جرام
(5) بنزويل مورفين (بيرونين) وأملاحه وكافة أكسيدات الأثير المورفينية الأخرى وأملاحها فيما عدا أثيل المورفين (ديونين):
(ديونين) ومثيل المورفين (كودايين) ................. 0.10 جرام
(6) ديهيدروديزوكسيمورفين (ديزومورفين) .... 0.06 جرام
(7) التبايين وأملاحه ...................... 0.15 جرام
(8) ز- أوكسيمورفين (جينو مورفين) ومركباته وكذا المركبات المورفينية الأخرى ذات الأزوت الخماسي التكافؤ ..... 0.20 جرام
(9) ديهيدرو أوكسيكودينون وأملاحه (كالايكودال) واستراته وأملاح الاسترات ..................... 0.06 جرام
ديهيدرو كودينون وأملاحه (كالديكوديد) واستراته وأملاح هذه الاسترات ...................... 0.06 جرام
ديهيدرومورفينون وأملاحه (كالديلوديد) واستراته وأملاح هذه الاسترات ...... 0.01 جرام
اسيتيلو ديهيدرو كودينون أو اسيتيلو ديميثيلو ديهيدروتباين وأملاحه (كالالاسيديكون) واسترات وأملاح هذه الاسترات ............. 0.06 جرام
ديهيدرو مورفين وأملاحه (كالبارا مورفان) واستراته وأملاح هذه الاسترات .................... 0.06 جرام
(10) الكوكايين وكافة أملاحه ........ 0.10 جرام
للاستعمال الباطني .............. 0.40 جرام
للاستعمال الظاهري ............. (X)
(11) الأكجونين وكافة أملاحه واستراته وأملاح هذه الاسترات ....... 0.10 جرام
(12) أستراثيلي لحمض ميثيل -1- فنيل -4 بيبريدين كاربوكسليك – 4 ز”بيتيدين” وجميع أملاحه ..... 0.65 جرام
(13) القنب “كانابيس ساتيفا” ............ 0.60 جرام
(×) بشرط أن يوصف في مركب لا تزيد نسبته فيه عن أربعة في المائة.
راتنج القنب................ 0.20 جرام
خلاصة القنب.................. 0.20 جرام
خلاصة القنب السائلة ........... 0.60 ملليمتر
صبغة القنب.............. 4.00 جرام
(14) ميثيل ديهيدرو مورفينون وأملاحه .... 0.30 جرام
(15) دي فينيل – 4، 4 دي مثيل أمينو – 6 دي فينيل -4، 4 (ومعروف أيضا تحت اسم دي مثيل أمينو – 6 دي فينيل 4، 4 هبتانون – 3 “ميتادون” وجميع أملاحه ........................ 0.125 جرام
(16) دي فينيل – 4، 4 مورفولينو – 6 هبتانون – 3 (ومعروف أيضا تحت اسم مورفولينو – دي فنيل – 4، 4 هبتانون -3 “فينادكسون” وجميع أملاحه ...... 0.250 جرام
وقد توضح قرين كل الحد الأقصى المسموح بصرفه في الوصفة الواحدة وهي:
(1) أمبول ماكسيتون (Maxiton amp) عدد 6 أمبول.
(2) أقراص ماكسيتون (Maxiton Tab) عدد30قرص.
(3) اقرص اكتدرون (Aktedron Tab) عدد 30قرص.
(4) أقراص دوريدين (Doriden Tap)عدد 30 قرص.
(28) مادة البنتازوسين وتحدد الكمية القصوى المصرح بصرفها بالواصفة الواحدة مائة وخمسون مللي جرام (150مللجرام ) أي ما يوازي خمسة أمبولات مسوسيجون على سبيل المثال.
 يضاف الى الجدول رقم (4) الملحق بالقانون الجواهر المخدرة الاتية وفقاً لما جاء بقرار وزير الصحة رقم 97 لسنة 1973 المنشور بالجريدة الرسمية العدد 83 في 16 / 4 / 1973.
 تدرج مادة البنتازوسين في الجدول رقم (4) تحت رقم (28) وفقاً لما جاء بقرار وزير الصحة رقم 167 لسنة 1977 المنشور بالوقائع المصرية العدد 91 في 19 / 4 / 1977.
 تعدل الفقرة (2) من الجدول رقم (4) وفقاً لما جاء بقرار وزير الصحة رقم 46 لسنة 1997 المنشور بالوقائع المصرية العدد 44 في 25 / 2 / 1997.
___________________________________________
الجدول رقم (5) 
النباتات الممنوع زراعتها
(1) القنب “كانابيس ساتيفا” ذكراً كان أو أنثى بجميع مسمياته مثل الحشيش – أو الكمنجة أو البانجو أو غير ذلك من الأسماء التي قد تطلق عليه.
(2) الخشخاش “باباقير سوميفيوم بجميع أصنافه ومسمياته مثل الأفيون أو أبو النوم أو غير ذلك من الأسماء التي قد تطلق عليه.
(3) جميع أنواع جنس البافير.
(4) الكوكا “ارثيروكسيلوم” بجميع أصنافه ومسمياته.
(5) القات بجميع أصنافه ومسمياته.
يجوز الترخيص للمصالح الحكومية والمعاهد العلمية بزراعة أي نبات من النباتات المبينة بالجدول رقم (5) وفقاً لما جاء بقرار وزير الصحة رقم 276 لسنة 1961 المنشور بالوقائع المصرية العدد 48 في 19 / 6 / 1961.
___________________________________________
الجدول رقم (6) 
أجزاء النباتات المستثناة من أحكام هذا القانون
(1) ألياف سيقان نبات القنب.
(2) بذور القنب المحموسة حمساً يكفل عدم إنباتها.
(3) بذور الخشخاش المحموسة حمساً يكفل عدم إنباتها.
_________________________________________

None
